# EDA – Simulación del proceso de matriculación vehicular (ANT) · Hito 2
**UIDE – Prácticas y Herramientas de Big Data.** Integrantes: Eduardo Pardo, James Romero, Ángel Paladines.

Este cuaderno es **reproducible**: con los dos CSV públicos en `data/raw/` ejecuta
`python src/generar_base_ant.py ...` y luego todas las celdas. Cada tabla y figura del informe sale de aquí
(`resultados/`).

In [1]:

import os, json, subprocess, sys, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RAW, SIN = f"{RAIZ}/data/raw", f"{RAIZ}/data/sintetico"
TAB, FIG = f"{RAIZ}/resultados/tablas", f"{RAIZ}/resultados/figuras"
os.makedirs(TAB, exist_ok=True); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "black",
                     "axes.prop_cycle": plt.cycler(color=["black", "0.45", "0.7"])})
HOY = pd.Timestamp("2026-11-22")
RES = {}                                   # resumen de cifras que usa el informe
def guardar(df, nombre): df.to_csv(f"{TAB}/{nombre}.csv", index=False); return df
print(pd.__version__, np.__version__)


3.0.5 2.5.3


## 1. Ficha técnica: volumen en disco y recuento exacto de registros

In [2]:

def contar_filas(ruta):
    n = 0
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 24), b""):
            n += bloque.count(b"\n")
    return n - 1                                       # sin encabezado
FUENTES = {
  "INEC vehículos matriculados 2019 (real)": (f"{RAW}/inec_vehiculos_matriculados_2019.csv", "https://www.ecuadorencifras.gob.ec/estadisticas-de-transporte-2019/"),
  "SRI vehículos nuevos 2026 (real)":        (f"{RAW}/sri_vehiculos_nuevos_2026.csv", "https://www.sri.gob.ec/datasets"),
}
for t in ["vehiculo", "propietario", "matricula", "revision_vehicular", "pagos", "multas"]:
    FUENTES[f"ANT sintética · {t.upper()}"] = (f"{SIN}/{t}.csv", "generado con src/generar_base_ant.py (semilla 2026)")
filas = []
for nombre, (ruta, url) in FUENTES.items():
    filas.append({"dataset": nombre, "filas": contar_filas(ruta), "columnas": len(pd.read_csv(ruta, nrows=0, sep=None, engine="python").columns),
                  "disco_mb": round(os.path.getsize(ruta) / 1e6, 1), "origen": url})
ficha = guardar(pd.DataFrame(filas), "01_ficha_tecnica")
sint = ficha[ficha.dataset.str.contains("sintética")]
RES["ficha"] = ficha.to_dict("records")
RES["sint_filas"] = int(sint.filas.sum()); RES["sint_mb"] = float(sint.disco_mb.sum())
ficha


,dataset,filas,columnas,disco_mb,origen
0,INEC vehículos matriculados 2019 (real),2311960,10,185.7,https://www.ecuadorencifras.gob.ec/estadistica...
1,SRI vehículos nuevos 2026 (real),509950,20,89.1,https://www.sri.gob.ec/datasets
2,ANT sintética · VEHICULO,2821910,21,542.5,generado con src/generar_base_ant.py (semilla ...
3,ANT sintética · PROPIETARIO,2211545,9,344.3,generado con src/generar_base_ant.py (semilla ...
4,ANT sintética · MATRICULA,2821910,9,292.8,generado con src/generar_base_ant.py (semilla ...
5,ANT sintética · REVISION_VEHICULAR,2565592,7,154.0,generado con src/generar_base_ant.py (semilla ...
6,ANT sintética · PAGOS,10319849,6,704.6,generado con src/generar_base_ant.py (semilla ...
7,ANT sintética · MULTAS,1032936,7,115.6,generado con src/generar_base_ant.py (semilla ...


## 2. Perfilado univariado (taxonomía de Abedjan et al., 2015)

In [3]:

V = pd.read_parquet(f"{SIN}/vehiculo.parquet")
ATOMICO = {  # tipo atómico (semántico) asignado tras revisar valores y patrones
 "id":"Entero · identificador (clave)", "placa":"Texto · código con patrón", "marca":"Categórico nominal",
 "modelo":"Texto libre (alta cardinalidad)", "anio":"Entero · ordinal temporal", "color":"Categórico nominal",
 "tipo":"Categórico binario", "clase":"Categórico nominal", "subcategoria":"Categórico nominal",
 "cilindraje":"Decimal · continuo (cm³)", "combustible":"Categórico nominal", "servicio":"Categórico nominal",
 "pasajeros":"Entero · discreto", "tonelaje":"Decimal · continuo (t)", "avaluo":"Decimal · moneda (USD)",
 "numero_chasis":"Texto · código 17 car.", "numero_motor":"Texto · código 11 car.", "provincia":"Categórico nominal",
 "canton":"Categórico nominal", "propietario_id":"Entero · clave foránea", "origen":"Categórico binario"}
def perfil(df, atom):
    out = []
    n = len(df)
    for c in df.columns:
        s = df[c]; nn = s.dropna(); vc = nn.value_counts()
        fila = {"columna": c, "tipo_atomico": atom.get(c, ""), "tipo_fisico": str(s.dtype),
                "nulos_%": round(100 * s.isna().mean(), 2), "distintos": int(nn.nunique()),
                "unicidad": round(nn.nunique() / max(len(nn), 1), 4),
                "constancia": round(vc.iloc[0] / max(len(nn), 1), 4) if len(vc) else np.nan,
                "moda": str(vc.index[0])[:22] if len(vc) else ""}
        if s.dtype.kind in "if":
            fila.update({"min": nn.min(), "mediana": nn.median(), "max": nn.max(), "asimetria": round(stats.skew(nn), 2)})
        else:
            L = nn.astype(str).str.len()
            fila.update({"min": L.min(), "mediana": L.median(), "max": L.max(), "asimetria": np.nan})
        out.append(fila)
    return pd.DataFrame(out)
perf_v = guardar(perfil(V, ATOMICO), "02_perfil_vehiculo")
perf_v


,columna,tipo_atomico,tipo_fisico,nulos_%,distintos,unicidad,constancia,moda,min,mediana,max,asimetria
0,id,Entero · identificador (clave),int32,0.00,2821910,1.0000,0.0000,1,1.00,1410955.50,2821910.0,0.000000
1,placa,Texto · código con patrón,str,0.00,2821910,1.0000,0.0000,AJ645Q,6.00,7.00,7.0,NaN
2,marca,Categórico nominal,str,0.00,338,0.0001,0.2354,CHEVROLET,2.00,6.00,22.0,NaN
3,modelo,Texto libre (alta cardinalidad),str,81.93,3513,0.0069,0.0253,SOLUTO LX AC 1.4 4P 4X,2.00,12.00,65.0,NaN
4,anio,Entero · ordinal temporal,int16,0.00,43,0.0000,0.0906,2027,1985.00,2014.00,2027.0,-0.520000
5,color,Categórico nominal,str,0.00,20,0.0000,0.2712,NEGRO,4.00,5.00,10.0,NaN
6,tipo,Categórico binario,str,0.00,2,0.0000,0.9480,LIVIANO,6.00,7.00,7.0,NaN
7,clase,Categórico nominal,str,0.00,11,0.0000,0.3051,MOTOCICLETA,3.00,9.00,11.0,NaN
8,subcategoria,Categórico nominal,str,81.93,50,0.0001,0.5758,PASEO,3.00,5.00,14.0,NaN
9,cilindraje,Decimal · continuo (cm³),float32,0.00,522,0.0002,0.1051,1368.0,0.00,1368.00,299800.0,15.530000


In [4]:

# Perfil de la fuente real INEC leída como TEXTO para ver los formatos "crudos"
I = pd.read_csv(f"{RAW}/inec_vehiculos_matriculados_2019.csv", sep=";", encoding="utf-8-sig", dtype=str)
hall = {
 "filas_duplicadas_%": round(100 * I.duplicated().mean(), 1),
 "TONELAJE_con_coma_decimal_sin_cero": int(I.TONELAJE.str.match(r"^,\d").sum()),
 "MODELO_censurado_2003_y_anteriores": int((I.MODELO == "2003 y Años Anteriores").sum()),
 "MARCA_OTRAS_MARCAS": int((I.MARCA == "OTRAS MARCAS").sum()),
 "TONELAJE_max": float(I.TONELAJE.str.replace(",", ".").astype(float).max()),
 "PASAJEROS_max": int(I.PASAJEROS.astype(int).max()),
}
perf_i = guardar(perfil(I, {}), "02_perfil_inec_texto")
RES["inec"] = hall; print(hall); del I
S = pd.read_csv(f"{RAW}/sri_vehiculos_nuevos_2026.csv", dtype=str, keep_default_na=False)
S["cil"] = S.CILINDRAJE.astype(float); S["comb"] = S["TIPO COMBUSTIBLE"]
RES["sri"] = {"COLOR2_vacio_%": round(100 * (S["COLOR 2"] == "").mean(), 2),
              "cil0_no_electricos": int(((S.cil == 0) & (S.comb != "ELECTRICO")).sum()),
              "cil_max": float(S.cil.max()), "filas_duplicadas": int(S.duplicated().sum())}
print(RES["sri"])


{'filas_duplicadas_%': np.float64(91.4), 'TONELAJE_con_coma_decimal_sin_cero': 1783583, 'MODELO_censurado_2003_y_anteriores': 455296, 'MARCA_OTRAS_MARCAS': 9336, 'TONELAJE_max': 744.0, 'PASAJEROS_max': 180}


{'COLOR2_vacio_%': np.float64(12.04), 'cil0_no_electricos': 31, 'cil_max': 299800.0, 'filas_duplicadas': 54077}


In [5]:

# Frecuencias y cardinalidad de las variables categóricas clave
cat = ["clase", "combustible", "servicio", "tipo", "origen", "provincia", "marca", "color"]
card = guardar(pd.DataFrame([{"variable": c, "cardinalidad": V[c].nunique(),
         "top_3": "; ".join(f"{k} ({v:.1%})" for k, v in V[c].value_counts(normalize=True).head(3).items())} for c in cat]),
         "02_cardinalidad")
freq_clase = guardar(V.clase.value_counts().rename_axis("clase").reset_index(name="n").assign(pct=lambda d: (100 * d.n / d.n.sum()).round(1)), "02_frecuencia_clase")
RES["clases"] = freq_clase.head(5).to_dict("records")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.4))
freq_clase.set_index("clase").n.div(1e6).plot.barh(ax=ax[0], color="0.3"); ax[0].invert_yaxis(); ax[0].set_xlabel("millones de vehículos"); ax[0].set_ylabel("")
ax[1].hist(V.anio, bins=np.arange(1984, 2029), color="0.3"); ax[1].set_xlabel("año del modelo"); ax[1].set_ylabel("vehículos")
plt.tight_layout(); plt.savefig(f"{FIG}/f1_univariado.png", dpi=200); card


,variable,cardinalidad,top_3
0,clase,11,MOTOCICLETA (30.5%); AUTOMOVIL (27.3%); SUV (1...
1,combustible,5,GASOLINA (85.1%); DIESEL (12.6%); HIBRIDO (1.7%)
2,servicio,4,PARTICULAR (92.1%); COMERCIAL (6.8%); ESTADO (...
3,tipo,2,LIVIANO (94.8%); PESADO (5.2%)
4,origen,2,PARQUE_INEC_2019 (81.9%); NUEVO_SRI_2026 (18.1%)
5,provincia,24,GUAYAS (24.0%); PICHINCHA (22.6%); MANABI (8.5%)
6,marca,338,CHEVROLET (23.5%); SUZUKI (6.1%); KIA (5.8%)
7,color,20,NEGRO (27.1%); BLANCO (21.6%); PLOMO (18.8%)


## 3. Auditoría de valores nulos y mecanismo de ausencia (MCAR / MAR / MNAR)

In [6]:

P = pd.read_parquet(f"{SIN}/propietario.parquet")
M = pd.read_parquet(f"{SIN}/matricula.parquet")
R = pd.read_parquet(f"{SIN}/revision_vehicular.parquet")
tablas = {"vehiculo": V, "propietario": P, "matricula": M, "revision_vehicular": R}
mat = pd.DataFrame({t: df.isna().mean() * 100 for t, df in tablas.items()}).T
mat = mat.loc[:, (mat > 0).any()]
guardar(mat.round(2).reset_index(names="tabla"), "03_matriz_nulos")
fig, ax = plt.subplots(figsize=(7.2, 1.9))
im = ax.imshow(mat.fillna(0).values, cmap="Greys", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=40, ha="right", fontsize=7)
ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index, fontsize=7)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.values[i, j]
        if not np.isnan(v) and v > 0: ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6, color="white" if v > 50 else "black")
plt.colorbar(im, ax=ax, label="% nulos"); plt.tight_layout(); plt.savefig(f"{FIG}/f2_matriz_nulos.png", dpi=200)
mat.round(2)


,apellidos,color,direccion,email,fecha_caducidad,fecha_matriculacion,fecha_nacimiento,kilometraje,modelo,observaciones,pasajeros,subcategoria,telefono,tonelaje
vehiculo,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,81.93,NaN,18.07,81.93,NaN,18.07
propietario,1.58,NaN,2.95,17.33,NaN,NaN,1.58,NaN,NaN,NaN,NaN,NaN,3.0,NaN
matricula,NaN,NaN,NaN,NaN,11.52,11.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
revision_vehicular,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.97,NaN,89.53,NaN,NaN,NaN,NaN


In [7]:

# Pruebas: ¿la ausencia depende de alguna variable OBSERVADA?
#  - categórica: chi-cuadrado sobre la tabla (nulo/no nulo) x categoría; tamaño de efecto = V de Cramér
#  - numérica: Mann-Whitney entre filas con y sin nulo; tamaño de efecto = AUC
def chi(ind, x):
    t = pd.crosstab(ind, x); c2, p, _, _ = stats.chi2_contingency(t)
    return p, np.sqrt(c2 / (t.values.sum() * (min(t.shape) - 1)))
def mw(ind, x):
    a, b = x[ind].dropna(), x[~ind].dropna(); u, p = stats.mannwhitneyu(a, b)
    return p, u / (len(a) * len(b))
rng = np.random.default_rng(1)
P["edad"] = (HOY - pd.to_datetime(P.fecha_nacimiento)).dt.days / 365.25
nat = P[P.tipo_persona == "NATURAL"].sample(400_000, random_state=1)
own = V.groupby("propietario_id").agg(canton=("canton", "first"), provincia=("provincia", "first"))
capitales = set(V[V.origen == "PARQUE_INEC_2019"].groupby("provincia").canton.agg(lambda s: s.value_counts().index[0]))
nat = nat.join(own, on="id"); nat["rural"] = ~nat.canton.isin(capitales)
r1all = R[R.intento == 1].merge(V[["id", "anio", "servicio", "clase"]], left_on="vehiculo_id", right_on="id")
r1all["edad_vehiculo"] = 2026 - r1all.anio
r1 = r1all.sample(400_000, random_state=1)
S2 = S.assign(c2=(S["COLOR 2"] == ""))
pruebas = []
def add(tabla, col, var, ind, x, kind):
    p, ef = chi(ind, x) if kind == "cat" else mw(ind, x)
    pruebas.append({"tabla": tabla, "columna": col, "contra": var, "p_valor": p, "efecto": round(ef, 3),
                    "medida": "V de Cramér" if kind == "cat" else "AUC"})
add("propietario", "telefono", "edad", nat.telefono.isna(), nat.edad, "num")
add("propietario", "telefono", "rural", nat.telefono.isna(), nat.rural, "cat")
add("propietario", "telefono", "provincia", nat.telefono.isna(), nat.provincia, "cat")
add("propietario", "email", "edad", nat.email.isna(), nat.edad, "num")
add("propietario", "email", "rural", nat.email.isna(), nat.rural, "cat")
add("propietario", "direccion", "edad", nat.direccion.isna(), nat.edad, "num")
add("propietario", "direccion", "rural (vía join)", nat.direccion.isna(), nat.rural, "cat")
add("revision_vehicular", "kilometraje", "edad del vehículo", r1.kilometraje.isna(), r1.edad_vehiculo, "num")
add("revision_vehicular", "kilometraje", "servicio", r1.kilometraje.isna(), r1.servicio, "cat")
add("vehiculo", "modelo", "origen", V.modelo.isna(), V.origen, "cat")
add("SRI real", "COLOR 2", "CLASE", S2.c2, S2.CLASE, "cat")
add("SRI real", "COLOR 2", "PAÍS", S2.c2, S2["PAÍS"], "cat")
pr = guardar(pd.DataFrame(pruebas), "03_pruebas_mecanismo")
pr.assign(p_valor=pr.p_valor.map(lambda p: f"{p:.2e}"))


,tabla,columna,contra,p_valor,efecto,medida
0,propietario,telefono,edad,9.58e-01,0.500,AUC
1,propietario,telefono,rural,9.07e-03,0.004,V de Cramér
2,propietario,telefono,provincia,5.85e-01,0.007,V de Cramér
3,propietario,email,edad,0.00e+00,0.898,AUC
4,propietario,email,rural,4.36e-01,0.001,V de Cramér
5,propietario,direccion,edad,7.53e-01,0.501,AUC
6,propietario,direccion,rural (vía join),0.00e+00,0.148,V de Cramér
7,revision_vehicular,kilometraje,edad del vehículo,0.00e+00,0.822,AUC
8,revision_vehicular,kilometraje,servicio,0.00e+00,0.272,V de Cramér
9,vehiculo,modelo,origen,0.00e+00,1.000,V de Cramér


In [8]:

# Indicio de MNAR en kilometraje (prueba con medición repetida):
# si un vehículo volvió a la RTV, el kilometraje del 2.º intento es un "testigo" de su recorrido real.
# Comparamos ese testigo entre vehículos cuyo 1.er kilometraje FALTÓ y los que sí lo tienen, dentro del mismo grupo de edad.
w = R.pivot_table(index="vehiculo_id", columns="intento", values="kilometraje", aggfunc="first", dropna=False)
w = w[w.index.isin(R[R.intento == 2].vehiculo_id)].join(V.set_index("id")[["anio", "clase"]])
w["edad"] = 2026 - w.anio
w = w[w[2].notna() & (w.edad >= 3)]
w["falto_1"] = w[1].isna()
w["km_anual_testigo"] = w[2] / w.edad
w["grupo_edad"] = pd.cut(w.edad, [2, 10, 20, 50])
q = w.groupby(["grupo_edad", "falto_1"], observed=True).km_anual_testigo.median().unstack().round(0)
q.columns = ["mediana_km_anual_si_hubo_dato", "mediana_km_anual_si_falto"]
pm = stats.mannwhitneyu(w.km_anual_testigo[w.falto_1], w.km_anual_testigo[~w.falto_1], alternative="greater")
RES["mnar"] = {"n_vehiculos": int(len(w)), "n_falto": int(w.falto_1.sum()), "p_valor": float(pm.pvalue),
               "auc": round(pm.statistic / (w.falto_1.sum() * (~w.falto_1).sum()), 3),
               "med_falto": float(w.km_anual_testigo[w.falto_1].median()), "med_dato": float(w.km_anual_testigo[~w.falto_1].median())}
guardar(q.reset_index().astype(str), "03_indicio_mnar_km")
print(RES["mnar"])
km = r1.kilometraje
imp = km.fillna(km.mean())
RES["km"] = {"nulos_%": round(100 * km.isna().mean(), 2), "media_obs": round(km.mean()), "sd_obs": round(km.std()),
             "sd_imputada_media": round(imp.std()), "reduccion_var_%": round(100 * (1 - imp.var() / km.var()), 1)}
print(RES["km"]); q


{'n_vehiculos': 50422, 'n_falto': 6104, 'p_valor': 0.0, 'auc': np.float64(0.79), 'med_falto': 18196.18709150327, 'med_dato': 11777.961842105264}
{'nulos_%': np.float64(7.71), 'media_obs': 163807, 'sd_obs': 176419, 'sd_imputada_media': 169483, 'reduccion_var_%': np.float64(7.7)}


,mediana_km_anual_si_hubo_dato,mediana_km_anual_si_falto
grupo_edad,,
"(2, 10]",10580.0,33854.0
"(10, 20]",12000.0,29334.0
"(20, 50]",11911.0,17003.0


In [9]:

# Clasificación final y estrategia (sin imputaciones ciegas)
clasif = pd.DataFrame([
 ["propietario.telefono", "3,0", "MCAR", "Efecto nulo: AUC con edad = 0,50 y V de Cramér ≤ 0,007 con zona y provincia.", "No imputar: es un dato de contacto; se deja vacío."],
 ["propietario.email", "17,3", "MAR", "Depende de la edad (AUC = 0,90): falta más en mayores.", "No imputar; si un modelo lo usa, crear indicador 'sin_email' y condicionar por edad."],
 ["propietario.direccion", "3,0", "MAR", "Depende de si el cantón es rural (V = 0,15; dato que existe vía join).", "Imputación condicional: completar con el cantón del vehículo (nivel cantón, no calle)."],
 ["revision.kilometraje", "≈8", "MNAR", "Prueba con medición repetida: quienes no tienen el 1.er dato recorren más km por año (hasta 3 veces más; AUC 0,79).", "No usar la media (sesgo hacia abajo). Indicador 'km_faltante' + mediana por clase y edad solo para describir; análisis de sensibilidad."],
 ["vehiculo.modelo / subcategoria", "81,9", "Estructural (MAR por origen)", "Falta en todo el parque INEC: la fuente no trae esa columna.", "No imputar; analizar el modelo solo en vehículos SRI."],
 ["vehiculo.pasajeros / tonelaje", "18,1", "Estructural (MAR por origen)", "Falta en todo vehículo SRI.", "Imputar con mediana por clase (dependencia clase → tonelaje)."],
 ["revision.observaciones", "≈90", "No aplica", "Vacío cuando el vehículo aprueba sin defectos.", "Reemplazar por 'SIN OBSERVACIONES'."],
 ["matricula.fecha_matriculacion", "11,5", "No aplica", "Vacío cuando el estado no es VIGENTE.", "Mantener nulo; se valida con la regla estado ↔ fecha."],
 ["SRI COLOR 2 (real)", "12,0", "MAR", "Depende de la clase y el país de origen.", "Imputar con COLOR 1: cuando hay dato, coinciden en el 98,8 %."],
 ["INEC 'OTRAS MARCAS' / SRI cilindraje = 0 (no eléctricos)", "0,4 / <0,01", "Nulos disfrazados", "Valores comodín que esconden un dato desconocido.", "Convertir a nulo y, en cilindraje, imputar con la mediana del mismo modelo."],
], columns=["columna", "nulos_%", "mecanismo", "evidencia", "estrategia"])
guardar(clasif, "03_clasificacion_nulos"); clasif


,columna,nulos_%,mecanismo,evidencia,estrategia
0,propietario.telefono,"3,0",MCAR,"Efecto nulo: AUC con edad = 0,50 y V de Cramér...",No imputar: es un dato de contacto; se deja va...
1,propietario.email,"17,3",MAR,"Depende de la edad (AUC = 0,90): falta más en ...","No imputar; si un modelo lo usa, crear indicad..."
2,propietario.direccion,"3,0",MAR,"Depende de si el cantón es rural (V = 0,15; da...",Imputación condicional: completar con el cantó...
3,revision.kilometraje,≈8,MNAR,Prueba con medición repetida: quienes no tiene...,No usar la media (sesgo hacia abajo). Indicado...
4,vehiculo.modelo / subcategoria,"81,9",Estructural (MAR por origen),Falta en todo el parque INEC: la fuente no tra...,No imputar; analizar el modelo solo en vehícul...
5,vehiculo.pasajeros / tonelaje,"18,1",Estructural (MAR por origen),Falta en todo vehículo SRI.,Imputar con mediana por clase (dependencia cla...
6,revision.observaciones,≈90,No aplica,Vacío cuando el vehículo aprueba sin defectos.,Reemplazar por 'SIN OBSERVACIONES'.
7,matricula.fecha_matriculacion,"11,5",No aplica,Vacío cuando el estado no es VIGENTE.,Mantener nulo; se valida con la regla estado ↔...
8,SRI COLOR 2 (real),"12,0",MAR,Depende de la clase y el país de origen.,"Imputar con COLOR 1: cuando hay dato, coincide..."
9,INEC 'OTRAS MARCAS' / SRI cilindraje = 0 (no e...,"0,4 / <0,01",Nulos disfrazados,Valores comodín que esconden un dato desconocido.,"Convertir a nulo y, en cilindraje, imputar con..."


## 4. Relaciones multivariadas, dependencias funcionales y outliers

In [10]:

PG = pd.read_parquet(f"{SIN}/pagos.parquet", columns=["vehiculo_id", "tipo", "valor", "fecha"],
                     filters=[("tipo", "==", "IMPUESTO_PROPIEDAD_SRI")])
MU = pd.read_parquet(f"{SIN}/multas.parquet", columns=["vehiculo_id", "tipo", "valor", "fecha", "estado"])
impp = PG[PG.tipo == "IMPUESTO_PROPIEDAD_SRI"].groupby("vehiculo_id").valor.sum().rename("impuesto")
mul = MU.groupby("vehiculo_id").valor.agg(n_multas="size", valor_multas="sum")
A = (V[["id", "anio", "cilindraje", "avaluo", "pasajeros", "tonelaje"]].set_index("id")
       .join(r1all.set_index("vehiculo_id").kilometraje).join(impp).join(mul))
A["edad"] = 2026 - A.pop("anio")
A[["n_multas", "valor_multas"]] = A[["n_multas", "valor_multas"]].fillna(0)
num = ["edad", "cilindraje", "avaluo", "impuesto", "kilometraje", "pasajeros", "tonelaje", "n_multas"]
B = A[num].dropna(); B = B.sample(min(600_000, len(B)), random_state=1)
pear, spear = B.corr("pearson"), B.corr("spearman")
guardar(pear.round(3).reset_index(names="variable"), "04_pearson"); guardar(spear.round(3).reset_index(names="variable"), "04_spearman")
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.0))
for a, m, t in [(ax[0], pear, "Pearson"), (ax[1], spear, "Spearman")]:
    a.imshow(m.values, cmap="Greys", vmin=-1, vmax=1)
    a.set_xticks(range(len(num))); a.set_xticklabels(num, rotation=60, ha="right", fontsize=6.5)
    a.set_yticks(range(len(num))); a.set_yticklabels(num if a is ax[0] else [], fontsize=6.5); a.set_title(t, fontsize=9)
    for i in range(len(num)):
        for j in range(len(num)):
            a.text(j, i, f"{m.values[i,j]:.2f}", ha="center", va="center", fontsize=5.2, color="white" if abs(m.values[i,j]) > .55 else "black")
plt.tight_layout(); plt.savefig(f"{FIG}/f3_correlaciones.png", dpi=200)
RES["corr"] = {"pear_av_imp": round(pear.loc["avaluo", "impuesto"], 3), "spear_av_imp": round(spear.loc["avaluo", "impuesto"], 3),
               "pear_edad_km": round(pear.loc["edad", "kilometraje"], 3), "spear_edad_km": round(spear.loc["edad", "kilometraje"], 3),
               "pear_edad_av": round(pear.loc["edad", "avaluo"], 3), "spear_edad_av": round(spear.loc["edad", "avaluo"], 3),
               "pear_pas_ton": round(pear.loc["pasajeros", "tonelaje"], 3), "spear_pas_ton": round(spear.loc["pasajeros", "tonelaje"], 3)}
RES["corr"]


{'pear_av_imp': np.float64(0.791),
 'spear_av_imp': np.float64(0.965),
 'pear_edad_km': np.float64(0.603),
 'spear_edad_km': np.float64(0.733),
 'pear_edad_av': np.float64(0.013),
 'spear_edad_av': np.float64(0.064),
 'pear_pas_ton': np.float64(0.255),
 'spear_pas_ton': np.float64(0.512)}

In [11]:

# Dependencias funcionales X -> Y: se cumple si cada valor de X tiene UN solo valor de Y
def fd(df, X, Y):
    g = df.groupby(X, observed=True)[Y].nunique()
    return int((g > 1).sum()), int(len(g))
S["AV"] = S["AVALÚO"]
dep = []
for nombre, df, X, Y in [
    ("vehiculo", V, "placa", "id"), ("vehiculo", V, "numero_chasis", "id"),
    ("vehiculo", V, "canton", "provincia"), ("vehiculo", V, "modelo", "marca"),
    ("SRI real", S, "CATEGORÍA", "MODELO"), ("SRI real", S, "CATEGORÍA", "AV"), ("SRI real", S, "MODELO", "MARCA"),
    ("vehiculo", V, "clase", "tipo")]:
    v_, n_ = fd(df, X, Y); dep.append({"tabla": nombre, "dependencia": f"{X} → {Y}", "grupos": n_, "violaciones": v_,
                                       "cumple": "sí" if v_ == 0 else ("casi" if v_ / n_ < .01 else "no")})
# avalúo -> impuesto (regla del SRI): se recalcula la tabla y se compara
def imp_sri(av):
    out = np.zeros_like(av)
    for lim, base, tasa in [(0,0,.005),(4000,20,.01),(8000,60,.02),(12000,140,.03),(16000,260,.04),(20000,420,.05),(24000,620,.06)]:
        m = av > lim; out[m] = base + (av[m] - lim) * tasa
    return np.round(out, 2)
chk = A[["avaluo", "impuesto"]].dropna().join(V.set_index("id").servicio)
chk = chk[chk.servicio != "ESTADO"]
dif = np.abs(imp_sri(chk.avaluo.values) - chk.impuesto.values) > .01
dep.append({"tabla": "vehiculo+pagos", "dependencia": "avaluo → impuesto (tabla SRI)", "grupos": len(chk), "violaciones": int(dif.sum()), "cumple": "sí" if dif.sum() == 0 else "no"})
dep = guardar(pd.DataFrame(dep), "04_dependencias_funcionales"); dep


,tabla,dependencia,grupos,violaciones,cumple
0,vehiculo,placa → id,2821910,0,sí
1,vehiculo,numero_chasis → id,2821910,0,sí
2,vehiculo,canton → provincia,209,2,casi
3,vehiculo,modelo → marca,3513,20,casi
4,SRI real,CATEGORÍA → MODELO,5799,0,sí
5,SRI real,CATEGORÍA → AV,5799,0,sí
6,SRI real,MODELO → MARCA,3513,20,casi
7,vehiculo,clase → tipo,11,9,no
8,vehiculo+pagos,avaluo → impuesto (tabla SRI),2578907,0,sí


In [12]:

# Outliers univariados: IQR (Tukey, k = 1,5) y Z-score (|z| > 3)
out = []
for c in ["avaluo", "cilindraje", "kilometraje", "tonelaje", "pasajeros", "impuesto"]:
    x = A[c].dropna(); q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
    z = (x - x.mean()) / x.std()
    out.append({"variable": c, "n": len(x), "lim_sup_IQR": round(q3 + 1.5 * iqr, 1), "outliers_IQR": int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()),
                "outliers_Z": int((z.abs() > 3).sum()), "max": round(x.max(), 1)})
out = guardar(pd.DataFrame(out).assign(pct_IQR=lambda d: (100 * d.outliers_IQR / d.n).round(2), pct_Z=lambda d: (100 * d.outliers_Z / d.n).round(2)), "04_outliers_univariados")
out


,variable,n,lim_sup_IQR,outliers_IQR,outliers_Z,max,pct_IQR,pct_Z
0,avaluo,2821910,7736.3,298142,40963,838800.0,10.57,1.45
1,cilindraje,2821910,3705.5,129701,72535,299800.0,4.60,2.57
2,kilometraje,2309541,538485.5,71022,30397,15302554.0,3.08,1.32
3,tonelaje,2311960,1.3,267114,48516,744.0,11.55,2.10
4,pasajeros,2311960,9.5,58702,27045,180.0,2.54,1.17
5,impuesto,2605843,39.0,370366,34065,49508.0,14.21,1.31


In [13]:

# Anomalías MULTIVARIADAS: el valor solo es raro en combinación con otras variables.
# 1) IQR condicional: kilometraje por año de uso, calculado DENTRO de cada clase.
A2 = A.join(V.set_index("id")[["clase", "servicio", "combustible", "origen"]])
A2["km_anual"] = A2.kilometraje / A2.edad.clip(lower=1)
def iqr_grupo(s):
    q1, q3 = s.quantile([.25, .75]); return (s > q3 + 3 * (q3 - q1))
A2["anom_km"] = A2.groupby("clase").km_anual.transform(iqr_grupo).fillna(False).astype(bool)
# 2) Z robusto del avalúo dentro de (clase, edad): |z| > 4 con mediana y MAD
g = A2.groupby(["clase", "edad"]).avaluo
med, mad = g.transform("median"), g.transform(lambda s: (s - s.median()).abs().median())
A2["z_rob_av"] = 0.6745 * (A2.avaluo - med) / mad.replace(0, np.nan)
A2["anom_av"] = A2.z_rob_av.abs() > 4
# 3) Reglas de negocio cruzadas
A2["moto_cil_alto"] = (A2.clase == "MOTOCICLETA") & (A2.cilindraje > 2000)
A2["nuevo_km_alto"] = (A2.origen == "NUEVO_SRI_2026") & (A2.kilometraje > 5000)
fm = M.set_index("vehiculo_id").fecha_matriculacion
p48 = PG[PG.tipo == "IMPUESTO_PROPIEDAD_SRI"].join(fm, on="vehiculo_id").dropna(subset=["fecha_matriculacion"])
viol48 = int(((pd.to_datetime(p48.fecha_matriculacion) - pd.to_datetime(p48.fecha)).dt.days < 2).sum())
mm = MU[(MU.tipo == "CONTRAVENCION") & (MU.estado == "PENDIENTE")].join(fm, on="vehiculo_id").dropna(subset=["fecha_matriculacion"])
fmm = pd.to_datetime(mm.fecha_matriculacion)
viol_multa = int(((fmm >= "2026-10-01") & (pd.to_datetime(mm.fecha) < fmm)).sum())
anom = guardar(pd.DataFrame([
  ["km por año extremo dentro de su clase (IQR k=3)", int(A2.anom_km.sum())],
  ["avalúo atípico para su clase y edad (Z robusto > 4)", int(A2.anom_av.sum())],
  ["motocicleta con cilindraje > 2.000 cm³", int(A2.moto_cil_alto.sum())],
  ["impuesto pagado < 48 h antes de matricular (regla Municipio)", viol48],
  ["matriculado desde oct-2026 con multa pendiente previa", viol_multa]],
  columns=["anomalia_multivariada", "casos"]), "04_anomalias_multivariadas")
RES["anom"] = anom.to_dict("records"); anom


,anomalia_multivariada,casos
0,km por año extremo dentro de su clase (IQR k=3),23437
1,avalúo atípico para su clase y edad (Z robusto...,69549
2,motocicleta con cilindraje > 2.000 cm³,415
3,impuesto pagado < 48 h antes de matricular (re...,784
4,matriculado desde oct-2026 con multa pendiente...,155


In [14]:

json.dump(RES, open(f"{TAB}/resumen_eda.json", "w"), ensure_ascii=False, indent=1, default=str)
print("resumen guardado")


resumen guardado
